In [1]:
import glob
import os
from pathlib import Path

import numpy as np
import json
import onnx
import tensorrt as trt
import pycuda.autoinit
import pycuda.driver as cuda
from IPython.display import display
from PIL import Image
from tqdm import tqdm

class TensorRTInference:
    def __init__(self, encoder_onnx="florence2_encoder_fp16.onnx", decoder_onnx="florence2_decoder_fp16.onnx", max_batch_size=8, max_new_tokens=16, rebuild=False):
        self.encoder_onnx = Path(encoder_onnx)
        self.decoder_onnx = Path(decoder_onnx)
        self.encoder_engine_path = self.encoder_onnx.with_suffix(".trt")
        self.decoder_engine_path = self.decoder_onnx.with_suffix(".trt")
        self.max_batch_size = max_batch_size
        self.max_new_tokens = max_new_tokens
        self.prompt = np.array([0, 2264, 16, 5, 2788, 11, 5, 2274, 116, 2], dtype=np.int64)
        self.bos_token_id = 0
        self.eos_token_id = 2
        self.mean = np.array([.485, .456, .406], np.float32)
        self.std = np.array([.229, .224, .225], np.float32)

        encoder_model = onnx.load(str(self.encoder_onnx), load_external_data=False)
        pixel_input = next(i for i in encoder_model.graph.input if i.name == "pixel_values")
        pixel_dims = pixel_input.type.tensor_type.shape.dim
        self.image_height = int(pixel_dims[-2].dim_value)
        self.image_width = int(pixel_dims[-1].dim_value)

        if rebuild or not self.encoder_engine_path.exists():
            self._build(self.encoder_onnx, self.encoder_engine_path, False)
        if rebuild or not self.decoder_engine_path.exists():
            self._build(self.decoder_onnx, self.decoder_engine_path, True)

        self.encoder, self.encoder_context, self.encoder_stream = self._load(self.encoder_engine_path)
        self.decoder, self.decoder_context, self.decoder_stream = self._load(self.decoder_engine_path)
        metadata = onnx.load(str(self.decoder_onnx), load_external_data=False).metadata_props
        blob = json.loads(next(p.value for p in metadata if p.key == "tokenizer_json"))
        meta = blob.pop("meta")
        from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_str(json.dumps(blob))
        self.special_ids = set(meta["special_ids"])

    def _build(self, onnx_path, engine_path, decoder):
        logger = trt.Logger(trt.Logger.WARNING)
        builder = trt.Builder(logger)
        flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
        network = builder.create_network(flags)
        parser = trt.OnnxParser(network, logger)

        if not parser.parse(Path(onnx_path).read_bytes()):
            errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
            raise RuntimeError("\n".join(errors))

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
        if builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
        profile = builder.create_optimization_profile()
        if decoder:
            profile.set_shape("decoder_input_ids", (1, 1), (self.max_batch_size, 1), (self.max_batch_size, self.max_new_tokens + 1))
            profile.set_shape("encoder_hidden_states", (1, 60, 768), (self.max_batch_size, 60, 768), (self.max_batch_size, 60, 768))
        else:
            prompt_length = len(self.prompt)
            profile.set_shape("input_ids", (1, prompt_length), (self.max_batch_size, prompt_length), (self.max_batch_size, prompt_length))
            image_shape = (3, self.image_height, self.image_width)
            profile.set_shape("pixel_values", (1, *image_shape), (self.max_batch_size, *image_shape), (self.max_batch_size, *image_shape))

        config.add_optimization_profile(profile)
        serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            raise RuntimeError(f"TensorRT build failed: {onnx_path}")
        Path(engine_path).write_bytes(serialized)
        print(f"created {engine_path}")

    def _load(self, path):
        runtime = trt.Runtime(trt.Logger(trt.Logger.ERROR))
        engine = runtime.deserialize_cuda_engine(Path(path).read_bytes())
        if engine is None:
            raise RuntimeError(f"Could not load {path}")
        return engine, engine.create_execution_context(), cuda.Stream()

    def _run(self, engine, context, stream, inputs, output_name):
        inputs = {name: np.ascontiguousarray(value) for name, value in inputs.items()}
        for name, value in inputs.items():
            context.set_input_shape(name, value.shape)
        shape = tuple(context.get_tensor_shape(output_name))
        dtype = np.dtype(trt.nptype(engine.get_tensor_dtype(output_name)))
        output = np.empty(shape, dtype=dtype)
        buffers = []
        for name, value in inputs.items():
            buffer = cuda.mem_alloc(value.nbytes)
            buffers.append(buffer)
            cuda.memcpy_htod_async(buffer, value, stream)
            context.set_tensor_address(name, int(buffer))
        output_buffer = cuda.mem_alloc(output.nbytes)
        buffers.append(output_buffer)
        context.set_tensor_address(output_name, int(output_buffer))
        if not context.execute_async_v3(stream.handle):
            raise RuntimeError("TensorRT inference failed")
        cuda.memcpy_dtoh_async(output, output_buffer, stream)
        stream.synchronize()
        return output

    def _preprocess(self, image):
        image = image.convert("RGB").resize((self.image_width, self.image_height), Image.Resampling.BICUBIC)
        pixels = np.asarray(image, np.float32) / 255.0
        pixels = (pixels - self.mean) / self.std
        return np.ascontiguousarray(pixels.transpose(2, 0, 1)[None], dtype=np.float16)

    def infer(self, images):
        batch = len(images)
        if not 1 <= batch <= self.max_batch_size:
            raise ValueError(f"batch must be between 1 and {self.max_batch_size}")
        pixels = np.concatenate([self._preprocess(image) for image in images])
        input_ids = np.tile(self.prompt[None], (batch, 1))
        hidden = self._run(self.encoder, self.encoder_context, self.encoder_stream, {"input_ids": input_ids, "pixel_values": pixels}, "encoder_hidden_states")
        generated = np.full((batch, self.max_new_tokens + 1), self.eos_token_id, dtype=np.int64)
        generated[:, 0] = self.bos_token_id
        finished = np.zeros(batch, dtype=bool)
        for length in range(1, self.max_new_tokens + 1):
            decoder_inputs = {"decoder_input_ids": generated[:, :length], "encoder_hidden_states": hidden}
            logits = self._run(self.decoder, self.decoder_context, self.decoder_stream, decoder_inputs, "logits")
            next_tokens = np.argmax(logits[:, -1, :], axis=-1).astype(np.int64)
            generated[:, length] = next_tokens
            finished |= next_tokens == self.eos_token_id
            if finished.all():
                break
        outputs = []
        for ids in generated:
            eos = np.flatnonzero(ids == self.eos_token_id)
            ids = ids[:eos[0]] if eos.size else ids
            ids = ids[~np.isin(ids, list(self.special_ids))]
            outputs.append(self.tokenizer.decode(ids.tolist()))
        return outputs


In [ ]:
images_path = glob.glob("./dataset/lp_crops_st_balanced/val/*.jpg")
model = TensorRTInference(encoder_onnx="florence2_encoder_fp16.onnx", decoder_onnx="florence2_decoder_fp16.onnx", max_batch_size=8, max_new_tokens=16)

for image_path in images_path[:5]:
    gt = Path(image_path).with_suffix(".txt").read_text().strip()
    image = Image.open(image_path).convert("RGB")
    pred = model.infer([image])[0]
    display(image)
    print("gt :", gt)
    print("pr :", pred)
    break

In [ ]:
BATCH_SIZE = 8
MAX_NEW_TOKENS = 16
model = TensorRTInference(encoder_onnx="florence2_encoder_fp16.onnx", decoder_onnx="florence2_decoder_fp16.onnx", max_batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS)

def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    previous = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        current = [i + 1]
        for j, c2 in enumerate(s2):
            current.append(min(previous[j + 1] + 1, current[j] + 1, previous[j] + (c1 != c2)))
        previous = current
    return previous[-1]

images_path = [p for p in glob.glob("./dataset/lp_crops_st_balanced/val/*.jpg") if os.path.exists(Path(p).with_suffix(".txt"))]
total = exact = distance_1 = distance_2 = 0

for start in tqdm(range(0, len(images_path), BATCH_SIZE)):
    batch_paths = images_path[start:start + BATCH_SIZE]
    batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
    predictions = model.infer(batch_images)

    for image_path, prediction in zip(batch_paths, predictions):
        ground_truth = Path(image_path).with_suffix(".txt").read_text().strip()
        distance = levenshtein_distance(ground_truth.lower(), prediction.replace(" ", "").strip().lower())
        total += 1
        exact += distance == 0
        distance_1 += distance <= 1
        distance_2 += distance <= 2

print(f"Total samples           : {total}")
print(f"Exact accuracy (dist=0) : {exact / total:.4f}")
print(f"Accuracy (dist<=1)      : {distance_1 / total:.4f}")
print(f"Accuracy (dist<=2)      : {distance_2 / total:.4f}")